# 14 — Independent confirmatory SD3 causal-mediation test

The notebook defines a fully new source/destination prompt manifest, nonoverlapping with the notebook 09 split. Sixteen object labels, three independent discovery templates and five confirmatory templates per label, 2 confirmatory seeds. The feature discovery, frozen ranking and subsequent inference use strictly disjoint prompt pairs. Pair/seed observations are not treated as independent labels.

In [ ]:
MODE="pilot"  # "pilot" checks two labels; "full" uses all 16
SEED=4107
DISCOVERY_PROMPTS_PER_LABEL = 1 if MODE=="pilot" else 3
CONFIRMATORY_PROMPTS_PER_LABEL = 1 if MODE=="pilot" else 5
LABEL_LIMIT = 2 if MODE=="pilot" else 16
EVALUATION_SEEDS = (1,) if MODE=="pilot" else (1,2)
MODEL_NAME = "topk_x4_k32"
FORCE_RERUN=False
# Fixed primary comparison: semantic_16 minus active_control_16 on target−replacement margin.


## 1. Freeze truly held-out source/destination pairs

Source and replacement prompts differ in exactly one target-object word. New prompts are rejected if they overlap previous notebook 09 prompts. Selection decisions are fixed before inspecting confirmatory scores.

In [ ]:
import hashlib,json,os,sys
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display

def root():
    wd=Path.cwd().resolve()
    for p in [wd,*wd.parents,Path("/home/jupyter/project/diffusion_attention_analysis_2")]:
        if (p/"src/diffusion_attention_analysis_v2").is_dir():return p
    raise FileNotFoundError("Project root unavailable")
ROOT=root();os.chdir(ROOT);sys.path.insert(0,str(ROOT/"src"))
from diffusion_attention_analysis_v2.config import load_config,resolve_path
from diffusion_attention_analysis_v2.sd3_notebook_utils import prepare_datashere_storage
from diffusion_attention_analysis_v2.sd3_counterfactual_v4 import new_heldout_manifest,run_campaign

storage=prepare_datashere_storage(project_root=ROOT,run_tag="sd3_medium_sae_circuit_150",
    container_name="container4",require_container=True,min_free_gb=12.)
assert storage["free_space_ok"]
cfg=load_config(ROOT/"configs/06_causal_analysis/sd3_medium_counterfactual_sae_mediation_v2.yaml")
previous=resolve_path(cfg,"data.output_dir",required=True)/"split_manifest.csv"
RUN=ROOT/"outputs/sd3_medium_sae_circuit_150"
OUT=RUN/"09_independent_confirmatory_v4"
OUT.mkdir(parents=True,exist_ok=True)
IDX=RUN/"07_sae_fidelity_rescue/checkpoint_index.json"
if not IDX.is_file():raise FileNotFoundError("Run notebook 12; checkpoint index unavailable")
CHECKPOINTS=json.loads(IDX.read_text())
checkpoint=Path(CHECKPOINTS[MODEL_NAME])
assert checkpoint.is_file(),checkpoint
candidate=new_heldout_manifest(previous,discovery_per_label=DISCOVERY_PROMPTS_PER_LABEL,
    confirm_per_label=CONFIRMATORY_PROMPTS_PER_LABEL,seed=SEED)
allowed=candidate.label.drop_duplicates().head(LABEL_LIMIT)
manifest=candidate[candidate.label.isin(allowed)].copy()
stable=manifest.to_csv(index=False)
sha=hashlib.sha256(stable.encode()).hexdigest()
manifest_path=OUT/"heldout_manifest.csv";fingerprint=OUT/"heldout_manifest.sha256"
if fingerprint.exists() and fingerprint.read_text().strip()!=sha:
    raise RuntimeError("Frozen held-out manifest changed; select a new run directory")
manifest.to_csv(manifest_path,index=False);fingerprint.write_text(sha)
assert manifest.pair_id.is_unique
assert not (set(manifest.loc[manifest.split.eq("discovery"),"pair_id"]) &
            set(manifest.loc[manifest.split.eq("confirmatory"),"pair_id"]))
display(manifest.head())
print("Labels",manifest.label.nunique(),"discovery",sum(manifest.split=="discovery"),
      "confirmatory",sum(manifest.split=="confirmatory"),"manifest SHA-256",sha)

## 2. Frozen test protocol and positive-control checks

Natural activation capture is generated for all new pairs. Discovery uses seed 0 only. Confirmatory uses seeds 1 and 2 in full mode. Targets are scored with Grounding DINO; downstream recovery, active-control dose and image preservation are measured on the same pairs. This can require substantial GPU time and disk; pilot mode is intended for preflight.

In [ ]:
assert torch.cuda.is_available(),"CUDA required for SD3 inference"
summary,primary=run_campaign(cfg,manifest,output=OUT/"experiment",
    reference_roots=[OUT/"experiment"],checkpoint=checkpoint,discovery_seed=0,
    evaluation_seeds=EVALUATION_SEEDS,max_eval_pairs=None,force=FORCE_RERUN)
display(summary)
display(pd.DataFrame([primary]))
(OUT/"frozen_analysis_plan.json").write_text(json.dumps({
    "manifest_sha256":sha,"checkpoint":str(checkpoint),"candidate":MODEL_NAME,
    "primary_estimand":"semantic_16 minus active_control_16 on paired target-replacement margin",
    "unit_of_independence":"target label; prompts and seeds nested under labels",
    "primary_ci":"label-cluster bootstrap 95 percent","seed":SEED,
    "discovery_seed":0,"evaluation_seeds":list(EVALUATION_SEEDS),
    "diagnostic_secondary":["full_patch_gate","target_effect","replacement_suppression",
                             "unchanged_context","ssim","downstream_probe_recovery"]
},indent=2))


## 3. Confirmatory analyses and interpretation limits

A positive full residual effect is a prerequisite for interpreting sparse mediation. Report target increase and replacement suppression separately. A positive target−replacement margin with low image quality or negative downstream recovery is not evidence of faithful object restoration.

In [ ]:
effects=pd.read_csv(OUT/"experiment/paired_effects.csv")
telemetry=pd.read_csv(OUT/"experiment/intervention_telemetry.csv")
doses=telemetry.groupby("arm").agg(mean_dose=("mean_dose","mean"),dose_vs_full=("dose_vs_full","mean"),
                                    probe_recovery=("probe_recovery","mean"))
display(doses)
if doses.loc["active_control_16","mean_dose"]<=0:
    raise RuntimeError("Inactive matched control: reject confirmatory experiment")
if (doses.loc["full","mean_dose"]<=0):
    raise RuntimeError("Full residual positive control has zero dose")
valid=effects[effects.site_valid]
table=valid.groupby("arm").agg(n=("margin_effect","size"),margin=("margin_effect","mean"),
    target=("target_effect","mean"),suppression=("replacement_suppression","mean"),
    context_shift=("context_shift","mean"),ssim=("ssim","mean"))
display(table)
table.to_csv(OUT/"valid_pairs_summary.csv")
print("Valid pair-seed observations",valid[valid.arm=="full"].shape[0],
      "of",effects[effects.arm=="full"].shape[0])
print("The full-split run and pilot must be treated as separate analyses. No significance claim is made from pilot mode.")